# Generative models

A generative model learns the underlying probability distribution of a dataset, which we call:

$$
p_{\text{model}}(x)
$$

This describes the probability density that the model assigns to a particular observation $x$.

- Probability density indicates how likely observations around a particular value are, with higher density meaning the model considers that region of the data space more likely.

The key idea is that the model learns how the data is distributed, rather than just learning a mapping from inputs to labels.

## Sampling the model

Once the model has learned the distribution, it can be used to generate new observations. 

$$
x_{\text{new}} \sim p_{\text{model}}(z)
$$

Here:

- $x_{\text{new}}$ is a newly generated observation.
- $\sim$ means "is sampled from".
- $p_{\text{model}}$ is the probability distribution learned by the model.
- $z$ represents the random latent variable or noise used to generate the observation.

When we sample from the model, it generates an observation according to its learned distribution.

- Each sample can be different because of the randomness involved.
- The generated observations are new samples, rather than copies of the training data.

## Density estimation

In addition to sampling, once the model has learned the data distribution it can be used for density estimation:

- Given an existing observation $x$, what probability density does the model assign to it?

$$
p_{\text{model}}(x)
$$

This is called density estimation because it is estimating the probability density associated with an observation.

It is useful for:

- Anomaly detection (identifying unusual observations).
- Understanding how well a model represents a dataset.
- Comparing how well different generative models fit data.

# Maximum likelihood estimation

Maximum likelihood estimation (MLE) is a method used to train generative models by finding the parameters that make the observed training data as likely as possible.

- A generative model has parameters, denoted by $\theta$, that determine the probability distribution it learns.
- MLE finds the optimal parameters, $\theta^*$, by maximising the likelihood of the observed training data under the model.

Mathematically:

$$
\theta^* = \arg\max_{\theta} p_{\text{model}}(X;\theta)
$$

- $\theta^*$: optimal model parameters.
- $\arg\max_{\theta}$: finds the parameter values that maximise the expression.
- $p_{\text{model}}(X;\theta)$: the likelihood of observing the entire dataset $X$ under the model with parameters $\theta$.

It uses the following process:

- Start with a model with parameters $\theta$.
- Evaluate how likely the observed training data is under the model.
- Adjust the parameters to increase the likelihood.
- Repeat until we find parameters that maximise the likelihood (or reach a suitable approximation).

# Cumulative distribution sampling

Cumulative distribution sampling is used to generate new samples from a probability distribution:

- It uses the cumulative distribution function (CDF) of the probability distribution
  - The CDF is the probability that a random variable $X$ takes a value less than or equal to a given value $x$
  - $$F_X(x)=P(X\leq x)$$
- The CDF is the integral of the probability density function (PDF)
  - The PDF describes how probability density is distributed across possible values of $x$
  - Maximum likelihood estimation can be used to estimate the parameters of a probability distribution from training data, defining the model's PDF.

Once the CDF is known, we can generate new samples using its inverse:

1. Sample a random value from a uniform distribution:
   $$z\sim U(0,1)$$
   This produces a random number between 0 and 1, where every value is equally likely.
2. Apply the inverse CDF:
   $$x_{\text{new}}=F_X^{-1}(z)$$
   The inverse CDF converts the uniformly distributed random number into a sample from our desired probability distribution.

The inverse CDF maps uniformly distributed values onto the desired distribution, accounting for its probability density.

# Flow models

A flow model transforms a simple probability distribution into a more complex one using a sequence of functions.

Each function must be invertible, meaning that we can reverse the transformation and recover the original input.

Invertibility allows us to:

- Transform samples from a simple distribution into more complex data.
- Reverse the transformation to recover the original input.
- Calculate the probability density of generated observations using the change of variables formula.

The key ideas are that:

- A flow model consists of a chain of invertible functions.
- Each function transforms the input into a new representation.
- The composition of invertible functions is itself invertible.
- This allows flow models to both generate new samples and calculate probability densities.

## Change of variables theorem

The change of variables theorem explains how to calculate the probability density of a transformed variable when we know the probability density of the original variable.

Suppose we start with a random variable $Z$, which follows a known probability distribution:
$$p_Z(z)$$

We then apply an invertible function to transform it into a new variable $X$:
$$x=f(z)$$

Because $f$ is invertible, we can also recover the original value:
$$z=f^{-1}(x)$$

The question is:

- If we know the probability density of $Z$, how do we calculate the probability density of $X$?

The change of variables formula is given as:
$$
p_X(x)=p_Z(f^{-1}(x))
\left|
\det\left(
\frac{\partial f^{-1}(x)}{\partial x}
\right)
\right|
$$

There are two main components:

1. Original probability density
   $$p_Z(f^{-1}(x))$$
   
   - We use the inverse function to find the original value $z$ corresponding to $x$.
   - We then evaluate the original probability density at that value.
2. Jacobian determinant
   $$\left|\det\left(\frac{\partial f^{-1}(x)}{\partial x}\right)\right|$$

   - This is called the absolute determinant of the Jacobian matrix.
   - It accounts for how the transformation stretches or compresses the space, ensuring that the probability density is adjusted correctly.
   - If the transformation stretches a region, its probability density decreases. If it compresses a region, its probability density increases.

The adjustment is necessary to ensure that the total probability remains equal to 1, even though the density changes during the transformation.

# Normalising flows

A normalising flow is a generative model that transforms samples from a simple, known probability distribution into samples from a more complex distribution using a sequence of invertible (bijective) transformations.

Suppose we start with a sample $z$ from a known probability distribution:
$$
z\sim p_Z(z)
$$

We then apply a chain of $K$ invertible transformations to obtain a new sample $x$:
$$
x=f_\theta(z)=f_K\circ\cdots\circ f_2\circ f_1(z)
$$

where:

- $f_i$ is an individual invertible transformation.
- $K$ is the number of transformations.
- $\theta$ represents the model parameters.
- Each transformation maps one intermediate representation to the next.

Because each transformation is invertible, we can recover the original sample $z$ from $x$ using the inverse transformations.


Using the change of variables theorem, we can calculate the probability density of the transformed sample:
$$
p_X(x)=p_Z(z)\prod_{i=1}^{K}
\left|
\det\left(
\frac{\partial f_i^{-1}}{\partial z_i}
\right)
\right|
$$

Taking the logarithm gives:
$$
\log p_X(x)=\log p_Z(z)+
\sum_{i=1}^{K}
\log\left|
\det\left(
\frac{\partial f_i^{-1}}{\partial z_i}
\right)
\right|
$$

where $z_i$ denotes the input to the $i$-th transformation.

- The first term, $\log p_Z(z)$, is the log probability density of the original sample.
- The summation accounts for how each transformation changes the probability density, using its Jacobian determinant.

## Triangular Jacobians

Normalising flows use a sequence of high-dimensional bijective transformations. To calculate probability densities using the change of variables theorem, we need to compute the determinant of each transformation's Jacobian matrix.

However, computing the determinant of a general $n\times n$ matrix can be computationally expensive.

To make this more efficient, normalising flows often restrict the transformations so that their Jacobian matrices are triangular:

- A Jacobian matrix is lower triangular if all entries above its main diagonal are zero:
- For a triangular matrix, the determinant is simply the product of the diagonal entries

Therefore, instead of calculating a general matrix determinant, we only need to multiply the diagonal entries.